# Source E: Medi-Cal FFS provider list

This notebook consolidates the statewide Medi-Cal FFS provider list to San Bernardino and Riverside counties, retains provider and geographic attributes, cleans the data, and saves a deduplicated Inland Empire extract.

Run the code cells from top to bottom with Shift+Enter.

## 1. Find the data and import pandas

In [ ]:
from pathlib import Path
import pandas as pd

relative_partition_dir = Path(
    "data/provider-side/E-mediCal-fee-for-service-enrolled-providers/"
    "zipped_bc_too_big_for_github"
)

repo_root = next(
    (
        folder
        for folder in (Path.cwd(), *Path.cwd().parents)
        if (folder / relative_partition_dir).is_dir()
    ),
    None,
)

if repo_root is None:
    raise FileNotFoundError(
        "Could not find the Source E data from the notebook's current folder: "
        f"{Path.cwd()}. Start Jupyter with the repository folder as its root."
    )

partition_dir = repo_root / relative_partition_dir
print("Source E folder:", partition_dir)


## 2. Load the two CSV parts

In [ ]:
csv_files = [
    partition_dir / "medi-cal-ffs-provider-listing_10_26_26_PART 1 first 200k line.csv",
    partition_dir / "medi-cal-ffs-provider-listing_10_26_26_PART 2 next 180k line.csv",
]

missing_files = [path for path in csv_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        "Missing expected Source E CSV file(s): "
        + ", ".join(str(path) for path in missing_files)
    )

providers_df = pd.concat(
    [
        pd.read_csv(
            path,
            encoding="cp1252",
            dtype=str,  # Preserve identifiers and leading zeros.
            low_memory=False,
            keep_default_na=False,
            na_values=[""],
        )
        for path in csv_files
    ],
    ignore_index=True,
)

required_columns = {"County", "CountyName", "State", "NPI", "Provider_Number"}
missing_columns = sorted(required_columns - set(providers_df.columns))
if missing_columns:
    raise ValueError(f"Source CSV is missing required columns: {missing_columns}")

print("Loaded both CSV parts into providers_df.")
print("Rows loaded:", len(providers_df))


## 3. Clean, subset to San Bernardino + Riverside, and deduplicate

In [ ]:
def normalize_text(value):
    if pd.isna(value):
        return ""
    return " ".join(str(value).replace("\u00a0", " ").split())

providers_df = providers_df.copy()

# Normalize text values and whitespace consistently.
for col in providers_df.columns:
    providers_df[col] = providers_df[col].map(normalize_text)

providers_df["County_norm"] = providers_df["County"].str.upper()
providers_df["CountyName_norm"] = providers_df["CountyName"].str.upper()
providers_df["State_norm"] = providers_df["State"].str.upper()

county_targets = {"SAN BERNARDINO", "RIVERSIDE"}
county_name_targets = {"SAN BERNARDINO COUNTY", "RIVERSIDE COUNTY"}
is_target_county = (
    providers_df["County_norm"].isin(county_targets)
    | providers_df["CountyName_norm"].isin(county_name_targets)
)
inland_empire_df = providers_df[is_target_county & providers_df["State_norm"].eq("CA")].copy()

print("Rows in statewide dataset:", len(providers_df))
print("Rows after California county filter:", len(inland_empire_df))
print("County values after filter:")
print(inland_empire_df["County"].value_counts(dropna=False))

# Use NPI when present, otherwise Provider_Number. Give records with neither ID
# a row-specific key so unrelated providers are not grouped together.
npi = inland_empire_df["NPI"].str.strip()
provider_number = inland_empire_df["Provider_Number"].str.strip()
inland_empire_df["provider_key"] = npi.mask(npi.eq(""), provider_number)
missing_provider_key = inland_empire_df["provider_key"].eq("")
inland_empire_df.loc[missing_provider_key, "provider_key"] = [
    f"UNKNOWN-{row_id}" for row_id in inland_empire_df.index[missing_provider_key]
]

# Build a normalized address key while preserving records with no usable address.
address_cols = ["Address", "Address2", "City", "State", "ZIP"]
address_parts = inland_empire_df[address_cols].apply(
    lambda column: column.str.upper().str.replace(r"\s+", " ", regex=True).str.strip()
)
inland_empire_df["address_key"] = address_parts.agg(" | ".join, axis=1)
missing_address = address_parts.eq("").all(axis=1)
inland_empire_df.loc[missing_address, "address_key"] = [
    f"UNKNOWN-ADDRESS-{row_id}" for row_id in inland_empire_df.index[missing_address]
]

duplicate_mask = inland_empire_df.duplicated(
    subset=["provider_key", "address_key"], keep=False
)
print("Rows in duplicate provider/address groups:", int(duplicate_mask.sum()))
if duplicate_mask.any():
    print("Sample duplicate records for review:")
    display(
        inland_empire_df.loc[duplicate_mask, ["Provider_Number", "NPI", *address_cols]]
        .head(10)
    )

# Keep the first row for exact provider/address duplicates; review the sample above
# if source ordering or conflicting duplicate attributes are a concern.
rows_before_deduplication = len(inland_empire_df)
inland_empire_df = inland_empire_df.drop_duplicates(
    subset=["provider_key", "address_key"], keep="first"
).copy()
print("Rows removed by provider+address deduplication:", rows_before_deduplication - len(inland_empire_df))

keep_cols = [
    "Provider_Number",
    "NPI",
    "NPI_Type",
    "Owner_Number",
    "Service_Location_Number",
    "Legal_Name",
    "Enroll_Status_Eff_DT",
    "Provider_Taxonomy",
    "NEMT_NMT_Provider_Type",
    "ANC_Provider_Type",
    "FI_Provider_Type_CD",
    "FI_Provider_Type",
    "Provider_License",
    "FI_Provider_Specialty_CD",
    "FI_Provider_Specialty",
    "Out_of_State_Indicator",
    "In_Out_State",
    "Address_Attention",
    "Address",
    "Address2",
    "City",
    "State",
    "ZIP",
    "ZIP_4",
    "DHCS_County_CD",
    "FIPS_County_CD",
    "County",
    "CountyName",
    "Phone_Number",
    "Medicaid_Patients",
    "CHIP_Patients",
    "Telehealth_Services",
    "Provider_Website",
    "Threshold_Languages",
    "Other_Languages",
    "Acc_Exam_Room",
    "Acc_Exterior_Building",
    "Acc_Interior_Building",
    "Acc_Parking",
    "Acc_Restroom",
    "Acc_Medical_Equipment",
    "Acc_Patient_Areas",
    "Acc_Patient_Diagnostic",
    "Provider_Source",
    "Billing_Provider",
    "Latitude",
    "Longitude",
]
missing_output_columns = sorted(set(keep_cols) - set(inland_empire_df.columns))
if missing_output_columns:
    raise ValueError(f"Source CSV is missing output columns: {missing_output_columns}")

inland_empire_clean = inland_empire_df[keep_cols].copy().reset_index(drop=True)
inland_empire_clean["County"] = inland_empire_clean["County"].replace({"": "Unknown"})
inland_empire_clean["CountyName"] = inland_empire_clean["CountyName"].replace({"": "Unknown"})

print("Final row count:", len(inland_empire_clean))
print("Rows by county:")
print(inland_empire_clean["County"].value_counts(dropna=False))
print("Top provider types:")
print(inland_empire_clean["FI_Provider_Type"].value_counts(dropna=False).head(10))
print("Missing/blank key-field counts:")
print(inland_empire_clean[["Provider_Number", "NPI", "Address", "City", "ZIP"]].eq("").sum())

assert inland_empire_df["State_norm"].eq("CA").all(), "Unexpected non-California rows found."
assert (
    inland_empire_df["County_norm"].isin(county_targets)
    | inland_empire_df["CountyName_norm"].isin(county_name_targets)
).all(), "Unexpected county rows found."

output_path = partition_dir / "san_bernardino_riverside_ffs_providers_clean.csv"
inland_empire_clean.to_csv(output_path, index=False)
print("Saved cleaned file to:", output_path)


## 4. Preview the clean Inland Empire table

In [ ]:
inland_empire_clean.head()
